In [1]:
import os
import re
import nltk
from nltk import Tree

import spacy
#spacy.cli.download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm") 

import pandas as pd
import numpy as np

import textdescriptives as td

from dataset_evaluation.utils import add_column
from dataset_evaluation.evaluation_framework import EvaluationFramework

import folia.main as folia
from pathlib import Path
from collections import defaultdict, Counter
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt
from datasets import load_dataset

import pylangacq

# Datasets

## Generated corpus (tinystories)

In [2]:
# generated corpus
tinystories = load_dataset('roneneldan/TinyStories', split='train[:1%]')
len(tinystories)

21197

In [3]:
df_tinystories = pd.DataFrame(tinystories['text'], columns=['story'])
tiny_tinystories = df_tinystories[:1200].copy()

## Narrative dataset (Nipopolou)

In [4]:
nico_path = '/Users/sabijn/Documents/PhD/Datasets/CHILDES_Nicolopoulou'
nicolopoulou = pylangacq.read_chat(nico_path)

df_narrative = pd.DataFrame({
    "text": [s.text() for s in nicolopoulou.stories()]
})

## Reference corpus (standard)

In [5]:
CC_dataset = load_dataset('codymd/cc100_en_sample')
CC_dataset_text = CC_dataset['train']['text']

In [6]:
ref_standard_dep = Path('datasets/CommonCrawl/CC_dep_lexicon_100000.csv')
ref_standard_uni = Path('datasets/CommonCrawl/CC_unigram_lexicon_100000.csv')
ref_standard_bi = Path('datasets/CommonCrawl/CC_bigram_lexicon_100000.csv')

## Reference corpus (spoken)

In [7]:
ref_spoken_b_csv = Path('/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/PeopleSpeech/PS_pos_bigram_100000.csv')
ref_spoken_u_csv  = Path('/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/PeopleSpeech/PS_pos_unigram_100000.csv')
ref_spoken_t_csv = Path('/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/PeopleSpeech/PS_pos_trigram_100000.csv')

# Evaluations

Linguistic annotations
- Syntactic tree depth per sentence according to the WNU_2022 paper (van Duijn & van Dijk, 2022)
- Average no. of complements per utterance
- Sentence complexity (words before the root)
- Lexical diversity
    - Standard TTR (not implemented).  
    Dividing the number of unique tokens (types) by the total number of tokens. 
    - msTTR (mean segmental type-token ratio (TTR)): 
    
    - maTTR (moving average TTR)
    - MTLD (average number of words for which a consecutive TTR is maintained).  
    Measure of Textual Lexical Diversity.
    - MTLD (moving)
    - MTLD (bidirectional)
    - HDD (bidirectional)
- Vendi score
- Dependency constrained perplexity
- Regular perplexity
- Past tense use
- MAUVE (diversity)

In [8]:
eval_f = EvaluationFramework(language='en',
                              pos_unigram=ref_spoken_u_csv, 
                              pos_bigram=ref_spoken_b_csv,
                              pos_trigram=ref_spoken_t_csv,
                              ref_unigram=ref_standard_uni,
                              ref_bigram=ref_standard_bi,
                              ref_ling_constrained=ref_standard_dep)

/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/PeopleSpeech/PS_pos_unigram_100000.csv


No sentence-transformers model found with name bert-base-uncased. Creating a new one with mean pooling.


## Syntatic tree depth
-- van Duijn & van Dijk (2022)

In [9]:
eval_f.add_pipe('syntactic_depth')

## Average no. of complements per utterance

In [10]:
eval_f.add_pipe('average_components')

## Sentence complexity
-- ChiSCor (2023) - Average number of words before the root

In [11]:
eval_f.add_pipe('wbr_average')

## Lexical diversity
-- ChiSCor (2023)

In [12]:
eval_f.add_pipe('lexical_diversity')

## Dependency distance
-- ChiSCor (2023)

In [13]:
eval_f.add_pipe('dependency_distance')

## Local contextuality

In [14]:
eval_f.add_pipe("local_contextuality")

## Grammaticality

$G(story) = \dfrac{\sum^k_{i=1}(G(s_i))}{k}$, with $s$ a sentence in the story.  
  
$G(sentence) = P(seq) = \log(\sqrt[n]{\prod_{i=1}^n \left( P(K_i) \right)})$.  
  
$P(K_i) = P(t_3t_2t_1) = \lambda_1 \cdot P(t_3|t_1t_2) + \lambda_2 \cdot P(t_3|t_2) + \lambda_3 \cdot P(t_3)$, where  
  
- where $\lambda_1 + \lambda_2 + \lambda_3 = 1$, ($\lambda_1 = 0.5, \lambda_2 = 0.3, \lambda_3 = 0.2 \rightarrow$ vadlapudi (2010))
- $P(t_3|t_1t_2) = \dfrac{f(t_1t_2t_3)}{f(t_1t_2)}$.  
- $P(t_3|t_2) = \dfrac{f(t_2t_3)}{f(t_2)}$.  
- $P(t_3) = \dfrac{f(t_3)}{\sum_{\forall t_i} f(t_i)}$


In [15]:
eval_f.add_pipe('grammaticality')

## Creative perplexity

In [16]:
eval_f.add_pipe('creative_perplexity_unigram')
eval_f.add_pipe('creative_perplexity_bigram')
eval_f.add_pipe('creative_perplexity_bigram_constrained')

## Vendi score
-- [Friedman (2023) The Vendi Score: A Diversity Evaluation Metric for Machine Learning](https://github.com/vertaix/Vendi-Score)

In [17]:
eval_f.add_pipe("vendi_ngram")
eval_f.add_pipe("vendi_embeddings")

## Run pipeline (real data)

In [ ]:
def load_or_run_eval(eval_f, dataset, column, path_name, *, run=False):
    if Path(path_name).exists() and not run:
        return pd.read_csv(path_name)

    dataset = eval_f.run_pipeline_on_df(dataset, column)
    return dataset

In [ ]:
# df_narrative = eval_f.run_pipeline_on_df(df_narrative, 'text')
# df_narrative = load_or_run_eval(eval_f, df_narrative, 'text', 'results/nicolopoulou_eval_results.csv')

Running syntactic_depth
Running average_components
Running wbr_average
Running lexical_diversity
Running dependency_distance
Running local_contextuality
Running grammaticality
Running creative_perplexity_unigram
Running creative_perplexity_bigram
Running creative_perplexity_bigram_constrained
Running vendi_ngram


/opt/miniconda3/envs/storylmdata/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Running vendi_embeddings


In [ ]:
def evaluate(text: str) -> float:
    sents = nlp(text).sents
    if len(list(sents)) < 2:
        print(len(sents))
        return 0.0

evaluate()

eval_f.run_component('vendi_ngram', df_narrative.iloc[0]['text'])

/opt/miniconda3/envs/storylmdata/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


0.0

## Run pipeline (tinystories)

In [19]:
tiny_tinystories = eval_f.run_pipeline_on_df(tiny_tinystories, 'story')

Running syntactic_depth
Running average_components
Running wbr_average
Running lexical_diversity
Running dependency_distance
Running local_contextuality
Running grammaticality
Running creative_perplexity_unigram
Running creative_perplexity_bigram
Running creative_perplexity_bigram_constrained
Running vendi_ngram


/opt/miniconda3/envs/storylmdata/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Running vendi_embeddings


In [31]:
tiny_tinystories.to_csv('results/tinystories_eval_results.csv')

## MAUVE
-- Pillutla (2021), see also Peeperkorn (NYP)

**Good to know**
- Mauve can have low scores due to lack of variety or poor quality.
- Adjust size of the histogram (num_buckets) based on the amount of trainingssamples (rule of thumb ``num_buckets < floor( #pred + #ref ) / 39)``).  
- Automatically runs gpt2
- Runs only with numpy 1.x not with numpy 2.x

**Usefull links**     
https://github.com/krishnap25/mauve.  
https://pypi.org/project/mauve-text/.  
https://huggingface-co.translate.goog/spaces/evaluate-metric/mauve?_x_tr_sl=en&_x_tr_tl=nl&_x_tr_hl=nl&_x_tr_pto=sc. 

In [20]:
# from evaluate import load

# mauve = load('mauve')
# p_text = CC_dataset_text[:1200]
# q_text = tiny_tinystories['story'].to_list()

# out = mauve.compute(
#     predictions=p_text,
#     references=q_text,
#     num_buckets=30,          
#     featurize_model_name="gpt2",   
#     device_id=-1,                  # force CPU featurization
#     max_text_length=256,           # default is 1024
#     kmeans_num_redo=1,             
#     kmeans_max_iter=100,           
#     pca_max_data=5000,            # cap PCA sample (use -1 to use all)
#     verbose=True
# )

# plt.plot(out.divergence_curve[:, 1], out.divergence_curve[:, 0])

In [21]:
# q_text = df_narrative['text'].to_list()

# out = mauve.compute(predictions=p_text, references=q_text, num_buckets=50)
# plt.plot(out.divergence_curve[:, 1], out.divergence_curve[:, 0])

## Vendi score overall

In [22]:
from vendi_score import text_utils

ngram_vs_nico = text_utils.ngram_vendi_score(list(map(str, df_narrative['text'].to_list())), ns=[1, 2, 3, 4])
ngram_vs_tiny = text_utils.ngram_vendi_score(list(map(str, tiny_tinystories['story'].to_list())), ns=[1, 2, 3, 4])
print('Vendi score Nicolopoulou:', ngram_vs_nico)
print('Vendi score TinyStories:', ngram_vs_tiny)

/opt/miniconda3/envs/storylmdata/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Vendi score Nicolopoulou: 212.83598433548977
Vendi score TinyStories: 425.5606300496613


## Vocabulary perplexity
-- code from ChiSCor (2023) but not published

### Bigram perplexity correct!

In [23]:
# def extract_PP_bigram(story, lexicon, alpha=1.0):
#     """
#     lexicon: DataFrame with columns ['lemma1','lemma2','freq'] where 'freq' is bigram count in the reference corpus
#     nlp: assumed to be defined (spaCy pipeline)
#     """
#     # Build bigram counts for the story
#     doc = nlp(story)
#     story_bigrams = [(doc[i].lemma_, doc[i+1].lemma_) for i in range(len(doc)-1)]
#     if not story_bigrams:
#         return None
#     story_counts = Counter(story_bigrams)

#     # Prep corpus counts: (lemma1) totals
#     hist_totals = lexicon.groupby('lemma1')['freq'].sum().rename('hist_total')
#      # |V(w_{i})| (unique lemma 2 with this lemma1)
#     V_by_hist = lexicon.groupby('lemma1')['lemma2'].nunique().rename('hist_vocab') 
    
#     # fallback vocabulary size per history for unseen lemma1's
#     global_V = lexicon['lemma2'].nunique()

#     # Fetch counts for the story bigrams from corpus (0 if unseen)
#     story_df = (
#         pd.DataFrame(story_counts.items(), columns=['bigram','count'])
#           .assign(lemma1=lambda df: df['bigram'].str[0],
#                   lemma2=lambda df: df['bigram'].str[1])
#           .drop(columns='bigram')
#           .merge(lexicon, how='left', on=['lemma1','lemma2'])
#     )
#     story_df['freq'] = story_df['freq'].fillna(0)

#     # Attach denominators and vocab sizes per history
#     story_df = (story_df
#         .merge(hist_totals, how='left', on='lemma1')
#         .merge(V_by_hist, how='left', on='lemma1'))

#     # For histories never seen in the corpus, use 0 for hist_total and a fallback vocab size
#     story_df['hist_total'] = story_df['hist_total'].fillna(0)
#     story_df['hist_vocab'] = story_df['hist_vocab'].fillna(global_V)

#     # Add-one smoothing and log-probs
#     # p(lemma2 | lemma1) = (c_bigram + alpha) / (c_history + alpha * V_history)
#     denom = story_df['hist_total'] + alpha * story_df['hist_vocab']
#     # Avoid zero denom if both are zero (edge case when global_V might be 0)
#     denom = denom.replace(0, np.finfo(float).tiny)

#     p = (story_df['freq'] + alpha) / denom
#     logp = np.log(p)

#     # 6) Perplexity over the full sequence (weighted by counts)
#     N = story_df['count'].sum()
#     avg_logp = (logp * story_df['count']).sum() / N
#     ppl = np.exp(-avg_logp)

#     return ppl

### Perplexity

In [24]:
# def extract_unigrams(story, nlp, *, with_pos=False,
#                      include_punct=False, include_space=False, exclude_pos=None) -> Counter:
#     """
#     Return Counter of (lemma,) or (lemma,pos) from the story.
#     """
#     doc = nlp(story)
#     bag = Counter()
#     for t in doc:
#         if not include_space and t.is_space: 
#             continue
#         if not include_punct and t.is_punct: 
#             continue
#         if exclude_pos and t.pos_ in exclude_pos:
#             continue
#         if with_pos:
#             bag[(t.lemma_, t.pos_)] += 1
#         else:
#             bag[(t.lemma_,)] += 1
#     return bag


In [25]:
# def compute_unigram_perplexity_from_counts(
#     story_counts: Counter,
#     lexicon: pd.DataFrame,          # columns = [*cols, 'freq']; e.g. ['lemma','freq'] or ['lemma','pos','freq']
#     *,
#     cols=("lemma",),                # or ("lemma","pos")
#     alpha: float = 1.0,
#     return_frame: bool = False,
# ):
#     """
#     Unigram PPL with Laplace smoothing:
#       p(w) = (c_corpus(w) + alpha) / (C + alpha*V)
#     Weighted by story token counts.
#     """
#     if not story_counts:
#         return (None, pd.DataFrame(columns=[*cols,"count","freq"])) if return_frame else None

#     story_df = pd.DataFrame([(*k, c) for k, c in story_counts.items()],
#                             columns=[*cols, "count"]) \
#                .merge(lexicon, how="left", on=list(cols))
#     story_df["freq"] = story_df["freq"].fillna(0).astype(np.int64)

#     C = int(lexicon["freq"].sum())                                # total tokens in reference
#     V = int(lexicon.drop(columns=["freq"]).drop_duplicates().shape[0]) or 1
#     denom = C + alpha * V

#     p = (story_df["freq"] + alpha) / denom
#     logp = np.log(p)

#     N = int(story_df["count"].sum())
#     ppl = float(np.exp(-(logp * story_df["count"]).sum() / N))

#     if return_frame:
#         out = story_df.copy()
#         out["prob"] = p
#         return ppl, out[[*cols, "count", "freq", "prob"]]
#     return ppl


In [26]:
# def extract_bigrams(story, nlp):
#     """
#     Return Counter of (lemma_{i-1}, lemma_i) from the story.
#     """
#     doc = nlp(story)
#     pairs = [(doc[i].lemma_, doc[i+1].lemma_) for i in range(len(doc)-1)]

#     return Counter(pairs)

# def _collect_dep_pairs(node, bag: Counter):
#     """
#     Collect linguistically constrained pairs.
#     Here: (child -> head) for selected relations.
#     Adjust the rules to your needs, but keep orientation consistent with lexicon.
#     """
#     # Verbal heads: subject/object relations
#     if node.pos_ in ('VERB', 'AUX'):
#         for child in node.children:
#             if child.dep_ in ('nsubj', 'nsubj:pass', 'obj'):
#                 bag[(child.lemma_, node.lemma_)] += 1

#     # Nominal heads: adjectival modifiers
#     if node.pos_ == 'NOUN':
#         for child in node.children:
#             if child.dep_ == 'amod' and child.pos_ == 'ADJ':
#                 bag[(child.lemma_, node.lemma_)] += 1

#     # Recurse
#     for child in node.children:
#         _collect_dep_pairs(child, bag)

# def extract_dep_pairs(story, nlp):
#     """
#     Return Counter of linguistically constrained (lemma1, lemma2) pairs.
#     Current orientation: (dependent lemma, head lemma).
#     """
#     bag = Counter()
#     for sent in nlp(story).sents:
#         _collect_dep_pairs(sent.root, bag)
#     return bag


# def compute_perplexity_from_counts(
#     story_counts: Counter,
#     lexicon: pd.DataFrame,
#     *,
#     alpha: float = 1.0,
#     vocab: str = "global",   # "global" or "per_history"
#     return_frame: bool = False
# ):
#     """
#     Compute perplexity for (lemma1, lemma2) counts using conditional bigram probabilities with Laplace smoothing.

#     lexicon: DataFrame with ['lemma1', 'lemma2', 'freq'] from the reference corpus.
#     alpha: Laplace smoothing strength (α=1 is add-one).
#     vocab:
#       - "global": use |{lemma2}| as V for all histories (textbook Laplace).
#       - "per_history": use |{lemma2: seen after lemma1}| as V(lemma1); falls back to global V when unknown.
#     return_frame: also return a diagnostic DataFrame with probs and components.
#     """
#     if not story_counts:
#         return (None, pd.DataFrame(columns=['lemma1','lemma2','count'])) if return_frame else None

#     # Story pairs + counts
#     story_df = (
#         pd.DataFrame([(l1, l2, c) for (l1, l2), c in story_counts.items()],
#                      columns=['lemma1','lemma2','count'])
#         .merge(lexicon, how='left', on=['lemma1','lemma2'])
#     )
#     story_df['freq'] = story_df['freq'].fillna(0)

#     # Denominators: totals per history (c(lemma1))
#     hist_totals = lexicon.groupby('lemma1')['freq'].sum().rename('hist_total')
#     story_df = story_df.merge(hist_totals, how='left', on='lemma1')
#     story_df['hist_total'] = story_df['hist_total'].fillna(0)

#     # Vocabulary sizes
#     global_V = int(lexicon['lemma2'].nunique()) or 1
#     if vocab == "per_history":
#         V_by_hist = lexicon.groupby('lemma1')['lemma2'].nunique().rename('hist_vocab')
#         story_df = story_df.merge(V_by_hist, how='left', on='lemma1')
#         story_df['hist_vocab'] = story_df['hist_vocab'].fillna(global_V)
#     else:
#         story_df['hist_vocab'] = global_V

#     # Smoothed conditional probabilities p(lemma2 | lemma1)
#     denom = story_df['hist_total'] + alpha * story_df['hist_vocab']

#     denom = denom.replace(0, np.finfo(float).tiny)  # Avoid log(0)
#     probs = (story_df['freq'] + alpha) / denom
#     logp = np.log(probs)

#     # Perplexity over total pair count (weights by story counts)
#     N = int(story_df['count'].sum())
#     if N == 0:
#         return (None, story_df) if return_frame else None

#     avg_logp = (logp * story_df['count']).sum() / N
#     ppl = float(np.exp(-avg_logp))

#     if return_frame:
#         out = story_df.copy()
#         out['prob'] = probs
#         return ppl, out[['lemma1','lemma2','count','freq','hist_total','hist_vocab','prob']]
    
#     return ppl


# # ---------- Simple orchestrator ----------

# def perplexity_for_creativity(
#     story: str,
#     lexicon: pd.DataFrame,
#     nlp,
#     *,
#     mode: str = "bigram",     # "unigram" | "bigram" | "dep"
#     alpha: float = 1.0,
#     vocab: str = "global",    # for bigrams only: "global" or "per_history"
#     with_pos_unigram: bool = False,   # unigram extractor option
#     unigram_cols = ("lemma",),        # or ("lemma","pos") to match your lexicon
#     return_frame: bool = False,
# ):
#     """
#     General entry point. Chooses the extractor and computes perplexity.
#     IMPORTANT: Your lexicon columns must match the extractor/cols:
#       - unigram_cols for 'unigram'
#       - ['lemma1','lemma2','freq'] for 'bigram'/'dep'
#     """
#     if mode == "bigram":
#         counts = extract_bigrams(story, nlp)
#         return compute_perplexity_from_counts(
#             counts, lexicon, alpha=alpha, vocab=vocab, return_frame=return_frame
#         )
#     elif mode == "dep":
#         counts = extract_dep_pairs(story, nlp)
#         return compute_perplexity_from_counts(
#             counts, lexicon, alpha=alpha, vocab=vocab, return_frame=return_frame
#         )
#     elif mode == "unigram":
#         counts = extract_unigrams(story, nlp, with_pos=with_pos_unigram)
#         return compute_unigram_perplexity_from_counts(
#             counts, lexicon, cols=unigram_cols, alpha=alpha, return_frame=return_frame
#         )
#     else:
#         raise ValueError("mode must be 'unigram', 'bigram', or 'dep'")



In [27]:
# for i in range(10):
#     print('*' * 10)
#     print(perplexity_for_creativity(tiny_tinystories.iloc[i]['story'], df_dp_lexicon_new, nlp, mode="dep", vocab="per_history"))
#     print(perplexity_for_creativity(tiny_tinystories.iloc[i]['story'], df_dp_lexicon_new, nlp, mode="dep", vocab="global"))

## Plotting and analysis

In [28]:
def plot_histogram(data, binsize=30, figure_size=(8,6), title='Dependency-constraint perplexities', xlabel='Perplexity', ylabel='Frequency'):
    plt.figure(figsize=figure_size)
    plt.hist(data, bins=binsize, edgecolor='black')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(axis='y', alpha=0.75)

In [29]:
# # Plot histograms
# plt.hist(tiny_tinystories['story_grammaticality'], bins=30, color='steelblue', alpha=0.6, label='Tinystories')
# plt.hist(df_narrative['story_grammaticality'], bins=30, color='darkorange', alpha=0.6, label='Childes')

# # Add labels and legend
# plt.xlabel('Grammaticality score')
# plt.ylabel('Frequency')
# plt.title('Grammaticality tinystories and childes')
# plt.legend()

# plt.show()